# 03. Validación cruzada repetida por grupos de 5 arquitecturas (PyTorch + timm)

Particiones estratificadas por grupos de secuencia de captura (3 repeticiones × 5 pliegues), entrenamiento en dos fases con cabecera común, métricas por pliegue y fuera de pliegue, DeLong, McNemar con corrección de Holm, bootstrap por grupos, figuras, complejidad computacional y Grad-CAM.

| Sección | Contenido |
|---|---|
| 0 | Configuración y entorno |
| 1 | Auditoría básica (MD5, pHash, DINOv2) y carga de los grupos finales (`grupos/grupos_finales.csv`) |
| 2 | Particiones |
| 3 | Control de atajos |
| 4–5 | Script de entrenamiento y lanzamiento con avance en vivo |
| 6 | Métricas por pliegue y calibración |
| 7 | Pruebas estadísticas sobre las predicciones fuera de pliegue |
| 8 | Figuras |
| 9 | Prueba de cambio de estrato (se omite si una clase tiene menos de 20 imágenes) |
| 10 | Complejidad computacional |
| 11 | Grad-CAM |
| 12 | Hiperparámetros y exportación |


In [ ]:
import os, json, time, sys, subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import timm

# ===================== CONFIGURACIÓN =====================
PROJECT_DIR = Path.cwd()                  # carpeta del notebook; debe contener dataT/
CSV_PATH = PROJECT_DIR / 'dataT' / 'data.csv'
assert CSV_PATH.exists(), 'No se encontró ' + str(CSV_PATH) + '. Ajusta PROJECT_DIR a la carpeta que contiene dataT/'
SMOKE_TEST = False                         # True: prueba rápida (1 época, 1 pliegue) en prueba/
OUT_DIR = PROJECT_DIR / ('prueba' if SMOKE_TEST else 'resultados')
GRUPOS = PROJECT_DIR / 'grupos' / 'grupos_finales.csv'   # agrupación por secuencia de captura
assert GRUPOS.exists(), 'Falta ' + str(GRUPOS)
CACHE_DIR = PROJECT_DIR / 'cache_256'     # copias redimensionadas (acelera la lectura)
GPUS = [0, 1]                             # índices de las dos A100
PROCS_POR_GPU = 6                         # trabajos simultáneos por GPU (cada uno usa 1-3 GB de los 80 GB)

print('Python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__,
      '| torch', torch.__version__, '(CUDA ' + str(torch.version.cuda) + ') | timm', timm.__version__)
assert torch.cuda.is_available() and torch.cuda.device_count() >= len(GPUS), (
    'PyTorch ve ' + str(torch.cuda.device_count()) + ' GPU. Usa el contenedor de docker compose (torch cu126).')
for i in GPUS:
    print('GPU', i, torch.cuda.get_device_name(i))

CFG = {
    'seed': 42,
    'n_folds': 5,
    'n_repeats': 3,                       # validación cruzada repetida: 3 × 5 pliegues
    'inner_val_splits': 7,                # 1/7 (~14%) del entrenamiento como validación interna
    'source_shift': True,                 # prueba de cambio de fuente (entrenar en un tamaño de imagen, probar en el otro)
    'img_size': 224,
    'cache_size': 256,
    'batch_size': 32,                     # igual en ambas fases
    'num_workers': 6,                     # procesos de carga por trabajo
    'drop_rate': 0.5,                     # dropout antes de la capa lineal de salida
    'weight_decay': 0.01,
    'bn_frozen': True,                    # BatchNorm en modo inferencia en ambas fases
    'phase1': {'epochs': 20, 'lr': 1e-3, 'patience': 5},   # backbone congelado
    'phase2': {'epochs': 40, 'lr': 1e-5, 'patience': 8},   # todo el modelo entrenable
    'plateau': {'factor': 0.5, 'patience': 3, 'min_lr': 1e-7},
    'aug': {'rrc_scale_min': 0.8, 'rotation_deg': 20, 'color_jitter': 0.2},
    'dedup': {'hamming': 6,               # distancia pHash máxima para casi-duplicados
              'cos': 0.95,                # similitud coseno mínima (DINOv2) para agrupar fotos parecidas
              'feat_models': ['vit_small_patch14_dinov2.lvd142m', 'convnext_tiny.fb_in1k']},
    'models': {
        'SwinT': 'swin_tiny_patch4_window7_224.ms_in1k',
        'ConvNeXtT': 'convnext_tiny.fb_in1k',
        'DeiT': 'deit_tiny_patch16_224.fb_in1k',
        'EfficientNetV2S': 'tf_efficientnetv2_s.in1k',
        'ResNet50': 'resnet50.tv_in1k',            # línea base clásica
    },
    # Los trabajos más largos primero
    'model_order': ['EfficientNetV2S', 'ConvNeXtT', 'SwinT', 'ResNet50', 'DeiT'],
    'class_names': ['Sana', 'BPR'],
}

# El /dev/shm de 64 MB que asigna Docker no alcanza para los workers del DataLoader
shm_gb = shutil.disk_usage('/dev/shm').total / 1024**3 if os.path.exists('/dev/shm') else 0.0
print('/dev/shm: %.2f GB | hilos de CPU: %d' % (shm_gb, os.cpu_count()))
if shm_gb < 2:
    CFG['num_workers'] = 0
    print('AVISO: /dev/shm pequeño; num_workers = 0. Usa el contenedor de docker compose (shm 16 GB).')
N_PROCS = len(GPUS) * PROCS_POR_GPU
if CFG['num_workers'] * N_PROCS > os.cpu_count():
    CFG['num_workers'] = max(1, os.cpu_count() // N_PROCS - 1)
    print('num_workers ajustado a', CFG['num_workers'], 'para no saturar la CPU')

if SMOKE_TEST:
    CFG['phase1']['epochs'] = 1
    CFG['phase2']['epochs'] = 1
    CFG['n_repeats'] = 1
    CFG['folds_to_run'] = [0]
else:
    CFG['folds_to_run'] = list(range(CFG['n_folds']))
CFG['procs_por_gpu'] = PROCS_POR_GPU
CFG['paths'] = {'project': str(PROJECT_DIR), 'csv': str(CSV_PATH), 'out': str(OUT_DIR), 'cache': str(CACHE_DIR)}

for sub in ['', 'ckpt', 'preds', 'logs', 'figs', 'tablas', 'hist', 'auditoria']:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CFG_PATH = OUT_DIR / 'config.json'
with open(CFG_PATH, 'w') as f:
    json.dump(CFG, f, indent=2)
print('Configuración guardada en', CFG_PATH)

## 0. Entorno computacional

In [ ]:
import platform
import importlib.metadata as md

def ver(p):
    try:
        return md.version(p)
    except Exception:
        return 'no instalado'

info = {
    'python': sys.version.split()[0],
    'sistema': platform.platform(),
    'torch': torch.__version__,
    'cuda_de_torch': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'paquetes': {p: ver(p) for p in ['torchvision', 'timm', 'grad-cam', 'scikit-learn', 'scipy', 'statsmodels',
                                     'numpy', 'pandas', 'Pillow', 'ImageHash', 'matplotlib', 'seaborn']},
    'gpus': [],
}
for i in range(torch.cuda.device_count()):
    pr = torch.cuda.get_device_properties(i)
    info['gpus'].append({'id': i, 'nombre': pr.name, 'memoria_GB': round(pr.total_memory / 1024**3, 1)})
try:
    salida = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
                            capture_output=True, text=True).stdout
    info['driver_nvidia'] = salida.strip().splitlines()[0]
except Exception:
    info['driver_nvidia'] = 'desconocido'
try:
    cpu = [l for l in open('/proc/cpuinfo') if l.startswith('model name')]
    info['cpu'] = cpu[0].split(':')[1].strip()
    info['hilos_cpu'] = len(cpu)
    mem = [l for l in open('/proc/meminfo') if l.startswith('MemTotal')][0]
    info['ram_GB'] = round(int(mem.split()[1]) / 1024**2, 1)
except Exception:
    pass
with open(OUT_DIR / 'env_info.json', 'w') as f:
    json.dump(info, f, indent=2)
print(json.dumps(info, indent=2, ensure_ascii=False))

## 1. Auditoría básica y grupos por secuencia de captura

La celda siguiente calcula MD5, pHash y la similitud DINOv2 entre las 2211 imágenes y muestra los pares más parecidos. Al final carga los grupos definitivos (`grupos/grupos_finales.csv`, obtenidos con los notebooks 01 y 02), que son los que se usan para formar las particiones: ningún grupo se reparte entre entrenamiento, validación y prueba.


In [ ]:
import hashlib
import imagehash
from concurrent.futures import ThreadPoolExecutor

df = pd.read_csv(CSV_PATH)
df['label'] = df['status'].astype(int)
df['abs_path'] = df['image_path'].apply(lambda p: str(PROJECT_DIR / p))
faltan = df[~df['abs_path'].apply(os.path.exists)]
assert len(faltan) == 0, 'Imágenes del CSV que no están en disco: ' + str(faltan['image_path'].tolist())
print('Imágenes:', len(df))
print(df['label'].value_counts().rename(index={0: 'Sana', 1: 'BPR'}).to_string())

def procesar(row):
    rel = os.path.splitext(row.image_path)[0] + '.png'
    dst = CACHE_DIR / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    with open(row.abs_path, 'rb') as fh:
        md5 = hashlib.md5(fh.read()).hexdigest()
    img = Image.open(row.abs_path).convert('RGB')
    w, h = img.size
    ph = str(imagehash.phash(img))
    if not dst.exists():
        img.resize((CFG['cache_size'], CFG['cache_size']), Image.Resampling.BICUBIC).save(dst)
    return md5, ph, w, h, str(dst)

with ThreadPoolExecutor(16) as ex:
    res = list(ex.map(procesar, df.itertuples()))
df['md5'], df['phash'], df['w'], df['h'], df['cache_path'] = zip(*res)
df['tamano'] = df['w'].astype(str) + 'x' + df['h'].astype(str)

# Estrato de resolución (los dos tamaños más frecuentes): control de atajos y sección 9
top = df['tamano'].value_counts()
print('\nTamaños originales más frecuentes:')
print(top.head().to_string())
fuentes = list(top.index[:2])
df['fuente'] = np.where(df['tamano'] == fuentes[0], 'A', np.where(df['tamano'] == (fuentes[1] if len(fuentes) > 1 else ''), 'B', 'otra'))
tab_fuente = pd.crosstab(df['fuente'], df['label'].map({0: 'Sana', 1: 'BPR'}), margins=True)
tab_fuente['% BPR'] = (100 * tab_fuente['BPR'] / tab_fuente['All']).round(1)
print('\nFuente A =', fuentes[0], '| Fuente B =', fuentes[1] if len(fuentes) > 1 else '-')
display(tab_fuente)
tab_fuente.to_csv(OUT_DIR / 'auditoria' / 'fuente_por_clase.csv')

In [ ]:
# 1) Distancia pHash entre todas las imágenes (0 = idénticas; 64 = totalmente distintas)
n = len(df)
hashes = np.array([int(h, 16) for h in df['phash']], dtype=np.uint64)
ham = np.zeros((n, n), dtype=np.uint8)
for i in range(0, n, 256):
    x = np.bitwise_xor(hashes[i:i + 256, None], hashes[None, :])
    ham[i:i + 256] = np.unpackbits(x.view(np.uint8), axis=-1).reshape(x.shape[0], n, 64).sum(-1)
np.fill_diagonal(ham, 255)

# 2) Similitud coseno entre descriptores DINOv2 (ConvNeXt de ImageNet si no se puede descargar)
import torchvision.transforms as T
from timm.data import resolve_data_config
dev0 = torch.device('cuda:0')
feat_model, fm = None, None
for nombre in CFG['dedup']['feat_models']:
    try:
        kw = {'img_size': 224} if 'dinov2' in nombre else {}
        fm = timm.create_model(nombre, pretrained=True, num_classes=0, **kw).eval().to(dev0)
        feat_model = nombre
        break
    except Exception as e:
        print('No se pudo cargar', nombre, '->', repr(e)[:120])
assert fm is not None, 'No se pudo cargar ningún modelo de características'
dc = resolve_data_config({}, model=fm)
tf_feat = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(dc['mean'], dc['std'])])
feats = []
with torch.no_grad():
    for i in range(0, n, 128):
        xb = torch.stack([tf_feat(Image.open(p).convert('RGB')) for p in df['cache_path'].iloc[i:i + 128]]).to(dev0)
        feats.append(torch.nn.functional.normalize(fm(xb).float(), dim=1).cpu())
feats = torch.cat(feats).numpy()
del fm
torch.cuda.empty_cache()
cos = feats @ feats.T
np.fill_diagonal(cos, -1)
usar_cos = 'dinov2' in feat_model
print('Modelo de características:', feat_model, '' if usar_cos else '(sin DINOv2: la agrupación usa solo pHash)')

# 3) Vecino más cercano de cada imagen
vec = pd.DataFrame({'ham_min': ham.min(1), 'cos_max': cos.max(1)})
vec['misma_clase_vecino_cos'] = df['label'].values == df['label'].values[cos.argmax(1)]
print('\nVecino más cercano (percentiles):')
print(vec[['ham_min', 'cos_max']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(vec['ham_min'], bins=range(0, 40), color='grey')
axes[0].axvline(CFG['dedup']['hamming'] + 0.5, color='red', ls='--', label='umbral de agrupación')
axes[0].set_xlabel('Distancia pHash al vecino más cercano'); axes[0].set_ylabel('Imágenes'); axes[0].legend()
axes[1].hist(vec['cos_max'], bins=50, color='grey')
if usar_cos:
    axes[1].axvline(CFG['dedup']['cos'], color='red', ls='--', label='umbral de agrupación')
    axes[1].legend()
axes[1].set_xlabel('Similitud coseno con el vecino más cercano (' + feat_model.split('.')[0] + ')')
plt.tight_layout(); fig.savefig(OUT_DIR / 'auditoria' / 'vecino_mas_cercano.png', dpi=200); plt.show()

# 4) Pares candidatos y grupos (unión de pares por pHash o por similitud de características)
iu = np.triu_indices(n, 1)
cand = (ham[iu] <= CFG['dedup']['hamming'])
if usar_cos:
    cand |= (cos[iu] >= CFG['dedup']['cos'])
pa, pb = iu[0][cand], iu[1][cand]
padre = list(range(n))
def raiz(i):
    while padre[i] != i:
        padre[i] = padre[padre[i]]
        i = padre[i]
    return i
for a, b in zip(pa, pb):
    ra, rb = raiz(a), raiz(b)
    if ra != rb:
        padre[rb] = ra
for _, idx in df.groupby('md5').groups.items():
    idx = list(idx)
    for b in idx[1:]:
        ra, rb = raiz(idx[0]), raiz(b)
        if ra != rb:
            padre[rb] = ra
df['group'] = [raiz(i) for i in range(n)]
tam = df.groupby('group').size()
limite = max(20, int(0.02 * n))
if tam.max() > limite:
    raise RuntimeError(('Un grupo reúne %d imágenes (límite %d): los umbrales de parecido son demasiado permisivos y '
                        'unirían mazorcas distintas. Sube CFG["dedup"]["cos"] o baja CFG["dedup"]["hamming"] y vuelve a '
                        'ejecutar desde la configuración.') % (tam.max(), limite))
print('\nDuplicados exactos (MD5):', int(len(df) - df['md5'].nunique()))
print('Pares agrupados: %d | grupos con más de una imagen: %d | imágenes en esos grupos: %d | grupos con etiquetas mezcladas: %d' % (
    len(pa), int((tam > 1).sum()), int(tam[tam > 1].sum()), int((df.groupby('group')['label'].nunique() > 1).sum())))

# 5) Los 40 pares más parecidos según cada criterio
orden_cos = np.argsort(-cos[iu])[:40]
orden_ham = np.argsort(ham[iu], kind='stable')[:40]
sel = np.unique(np.r_[orden_cos, orden_ham])
tabla = pd.DataFrame({'i': iu[0][sel], 'j': iu[1][sel]})
tabla['hamming'] = ham[tabla['i'], tabla['j']]
tabla['coseno'] = cos[tabla['i'], tabla['j']].round(4)
tabla['img_i'] = df['image_path'].values[tabla['i']]
tabla['img_j'] = df['image_path'].values[tabla['j']]
tabla['clase_i'] = df['label'].values[tabla['i']]
tabla['clase_j'] = df['label'].values[tabla['j']]
tabla['agrupado'] = df['group'].values[tabla['i']] == df['group'].values[tabla['j']]
tabla = tabla.sort_values('coseno', ascending=False)
tabla.to_csv(OUT_DIR / 'auditoria' / 'pares_mas_parecidos.csv', index=False)

def mostrar_pares(filas, titulo, archivo):
    filas = filas.head(12)
    if not len(filas):
        return
    fig, axes = plt.subplots(len(filas), 2, figsize=(5, 2.4 * len(filas)))
    axes = np.atleast_2d(axes)
    for r, (_, t) in enumerate(filas.iterrows()):
        for c, k in enumerate([t['i'], t['j']]):
            axes[r, c].imshow(Image.open(df.loc[k, 'cache_path']))
            axes[r, c].set_title('%s | %s' % (os.path.basename(df.loc[k, 'image_path']), CFG['class_names'][int(df.loc[k, 'label'])]), fontsize=6)
            axes[r, c].axis('off')
        axes[r, 0].text(-0.05, 0.5, 'd=%d\ncos=%.3f\n%s' % (t['hamming'], t['coseno'], 'AGRUPADO' if t['agrupado'] else ''),
                        transform=axes[r, 0].transAxes, ha='right', va='center', fontsize=6)
    fig.suptitle(titulo, fontsize=8)
    plt.tight_layout(); fig.savefig(OUT_DIR / 'auditoria' / archivo, dpi=150); plt.show()

mostrar_pares(tabla, 'Pares más parecidos (revisar si son la misma mazorca)', 'pares_mas_parecidos.png')
print('Tabla completa en', OUT_DIR / 'auditoria' / 'pares_mas_parecidos.csv')

# ===== Grupos por secuencia de captura (notebooks 01 y 02) =====
gfin = pd.read_csv(GRUPOS)
mapa = dict(zip(gfin['image_path'], gfin['group']))
assert df['image_path'].isin(mapa).all() and len(gfin) == len(df), 'grupos_finales.csv no coincide con data.csv'
df['group_basico'] = df['group']
df['group'] = df['image_path'].map(mapa).astype(int)
tamg = df.groupby('group').size()
print('\nGrupos por secuencia de captura: %d (solo pHash y DINOv2 >= 0.95: %d) | imágenes en grupos de más de una: %d | grupo mayor: %d (%.1f %%)' % (
    len(tamg), df['group_basico'].nunique(), int(tamg[tamg > 1].sum()), tamg.max(), 100 * tamg.max() / len(df)))


## 2. Particiones

Tres repeticiones de 5 pliegues estratificados por grupo, cada una con su semilla. La prueba se usa solo en la evaluación final; 1/7 del entrenamiento queda como validación interna (detención temprana, tasa de aprendizaje y umbral). Si `folds.csv` ya existe con la misma configuración, se reutiliza.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

FOLDS_CSV = OUT_DIR / 'folds.csv'
JOBS_JSON = OUT_DIR / 'jobs.json'
firma = {'n': len(df), 'dedup': CFG['dedup'], 'n_repeats': CFG['n_repeats'], 'n_folds': CFG['n_folds'],
         'seed': CFG['seed'], 'grupos': int(df['group'].nunique()), 'agrupacion': 'secuencia_captura', 'source_shift': CFG['source_shift']}
hay_resultados = any((OUT_DIR / 'preds').glob('*.done'))
if FOLDS_CSV.exists() and (OUT_DIR / 'folds_firma.json').exists() and json.load(open(OUT_DIR / 'folds_firma.json')) == firma:
    df = pd.read_csv(FOLDS_CSV)
    print('Se reutilizan las particiones guardadas en', FOLDS_CSV)
else:
    assert not hay_resultados, ('La configuración de particiones cambió pero ya hay resultados en ' + str(OUT_DIR) +
                                '. Usa otra carpeta de salida o bórrala.')
    y_all, g_all = df['label'].values, df['group'].values

    def asignar_val(col, tr_idx, semilla):
        inner = StratifiedGroupKFold(n_splits=CFG['inner_val_splits'], shuffle=True, random_state=semilla)
        _, va_rel = next(inner.split(tr_idx, y_all[tr_idx], g_all[tr_idx]))
        df.loc[tr_idx, col] = 'train'
        df.loc[tr_idx[va_rel], col] = 'val'

    # Validación cruzada repetida: cada repetición usa otra semilla para formar los pliegues
    for r in range(CFG['n_repeats']):
        outer = StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'] + 100 * r)
        for k, (tr, te) in enumerate(outer.split(df, y_all, g_all)):
            col = 'split_r%d_f%d' % (r, k)
            df[col] = 'test'
            asignar_val(col, tr, CFG['seed'] + 100 * r + k)

    # Cambio de estrato: entrenar con un tamaño de imagen y probar con el otro (sin grupos mixtos)
    if CFG['source_shift']:
        mezcla = df.groupby('group')['fuente'].transform('nunique') > 1
        for ent, pru in (('A', 'B'), ('B', 'A')):
            col = 'split_src_' + ent
            df[col] = 'excluida'
            tr = np.where((df['fuente'] == ent) & ~mezcla)[0]
            te = np.where((df['fuente'] == pru) & ~mezcla)[0]
            ok = all(min((y_all[ix] == c).sum() for c in (0, 1)) >= 20 for ix in (tr, te))
            if not ok:
                print('Cambio de fuente %s->%s omitido: alguna clase tiene menos de 20 imágenes' % (ent, pru))
                df.drop(columns=col, inplace=True)
                continue
            df.loc[te, col] = 'test'
            asignar_val(col, tr, CFG['seed'] + 500 + ord(ent))

    cols = [c for c in df.columns if c.startswith('split_')]
    for c in cols:
        s = df[df[c] != 'excluida'].groupby('group')[c].nunique()
        assert (s == 1).all(), 'Fuga de grupos en ' + c
    df.to_csv(FOLDS_CSV, index=False)
    json.dump(firma, open(OUT_DIR / 'folds_firma.json', 'w'), indent=2)
    print('Particiones guardadas en', FOLDS_CSV)

# Lista de trabajos (modelo, partición, semilla)
split_cols = [c for c in df.columns if c.startswith('split_')]
cv_splits = ['r%d_f%d' % (r, k) for r in range(CFG['n_repeats']) for k in CFG['folds_to_run']]
src_splits = [c[len('split_'):] for c in split_cols if c.startswith('split_src_')] if CFG['source_shift'] else []
jobs = []
for s in cv_splits + src_splits:
    if s.startswith('r'):
        r, k = [int(x) for x in s[1:].split('_f')]
        semilla = CFG['seed'] + 1000 * r + k
    else:
        semilla = CFG['seed'] + 500 + ord(s[-1])
    for m in CFG['model_order']:
        jobs.append({'model': m, 'split': s, 'tag': m + '_' + s, 'seed': semilla})
json.dump(jobs, open(JOBS_JSON, 'w'), indent=1)
print('Trabajos a entrenar:', len(jobs), '(%d de validación cruzada + %d de cambio de fuente)' % (
    len(cv_splits) * len(CFG['models']), len(src_splits) * len(CFG['models'])))

conteos = []
for c in split_cols:
    for sp, nombre in [('train', 'Entrenamiento'), ('val', 'Validación'), ('test', 'Prueba')]:
        sub = df[df[c] == sp]
        conteos.append({'Partición': c[len('split_'):], 'Conjunto': nombre, 'Sana': int((sub['label'] == 0).sum()),
                        'BPR': int((sub['label'] == 1).sum()), 'Total': len(sub),
                        '% BPR': round(100 * sub['label'].mean(), 1) if len(sub) else np.nan})
tabla_conteos = pd.DataFrame(conteos)
tabla_conteos.to_csv(OUT_DIR / 'tablas' / 'tabla_conteos_particiones.csv', index=False)
display(tabla_conteos[tabla_conteos['Partición'].str.startswith(('r0_', 'src_'))])

## 3. Control de atajos

Regresión logística con rasgos ajenos a la lesión (tamaño de la imagen original, color del borde y color global), evaluada con los mismos pliegues que las redes.


In [ ]:
# Regresión logística con rasgos ajenos a la lesión
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def rasgos(ruta):
    a = np.asarray(Image.open(ruta).convert('RGB').resize((128, 128)), dtype=np.float32) / 255.0
    hsv = np.asarray(Image.fromarray((a * 255).astype(np.uint8)).convert('HSV'), dtype=np.float32) / 255.0
    borde = np.concatenate([a[:13].reshape(-1, 3), a[-13:].reshape(-1, 3), a[:, :13].reshape(-1, 3), a[:, -13:].reshape(-1, 3)])
    centro = a[40:88, 40:88].reshape(-1, 3)
    hist = np.concatenate([np.histogram(hsv[..., c], bins=8, range=(0, 1))[0] / hsv[..., c].size for c in range(3)])
    return np.r_[a.reshape(-1, 3).mean(0), a.reshape(-1, 3).std(0), hist, borde.mean(0), borde.std(0), centro.mean(0)]

R = np.stack([rasgos(p) for p in df['cache_path']])
nombres = (['media_R', 'media_G', 'media_B', 'de_R', 'de_G', 'de_B'] + ['hist_%s%d' % (c, b) for c in 'HSV' for b in range(8)] +
           ['borde_R', 'borde_G', 'borde_B', 'borde_de_R', 'borde_de_G', 'borde_de_B', 'centro_R', 'centro_G', 'centro_B'])
R = pd.DataFrame(R, columns=nombres)
R['ancho_px'] = df['w'].values
R['fuente_B'] = (df['fuente'] == 'B').astype(float).values
conjuntos = {
    'Tamaño de la imagen original (fuente)': ['ancho_px', 'fuente_B'],
    'Color del borde (fondo)': [c for c in nombres if c.startswith('borde')],
    'Color global (medias, DE e histograma HSV)': [c for c in nombres if c.startswith(('media', 'de_', 'hist'))],
    'Todos los rasgos anteriores': list(R.columns),
}
y_all = df['label'].values
filas = []
for nombre, cols in conjuntos.items():
    oof = np.zeros(len(df))
    for k in range(CFG['n_folds']):
        col = 'split_r0_f%d' % k
        if col not in df:
            continue
        tr, te = (df[col] != 'test').values, (df[col] == 'test').values
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
        clf.fit(R.loc[tr, cols], y_all[tr])
        oof[te] = clf.predict_proba(R.loc[te, cols])[:, 1]
    evaluadas = np.zeros(len(df), bool)
    for k in range(CFG['n_folds']):
        if 'split_r0_f%d' % k in df:
            evaluadas |= (df['split_r0_f%d' % k] == 'test').values
    filas.append({'Rasgos': nombre, 'N rasgos': len(cols), 'AUC (pliegues de la repetición 1)': round(roc_auc_score(y_all[evaluadas], oof[evaluadas]), 3)})
tabla_atajos = pd.DataFrame(filas)
tabla_atajos.to_csv(OUT_DIR / 'auditoria' / 'control_atajos.csv', index=False)
display(tabla_atajos)

## 4. Script de entrenamiento

Se escribe a disco y se ejecuta como procesos independientes. Protocolo idéntico para los 5 modelos:

- Cabecera: pooling global propio de cada arquitectura + Dropout(0.5) + capa lineal de 1 salida, inicializada igual en los 5 modelos (normal truncada, DE 0.02; sesgo 0). Pérdida: entropía cruzada binaria sobre el logit.
- Fase 1: backbone congelado, AdamW lr = 1e-3. Fase 2: todo el modelo entrenable, AdamW lr = 1e-5, partiendo de los mejores pesos de la fase 1.
- BatchNorm en modo inferencia (estadísticas de ImageNet) en ambas fases; sus parámetros afines se ajustan en la fase 2.
- ReduceLROnPlateau (factor 0.5, paciencia 3) y detención temprana sobre la pérdida de validación; se restauran los mejores pesos.
- Aumento de datos: recorte aleatorio (80–100% del área), volteo horizontal, rotación ±20°, variación de brillo/contraste/saturación ±20%.
- Normalización con la media y desviación de ImageNet de cada modelo; bfloat16 en entrenamiento y float32 en la predicción final.
- Umbral de validación: entre los umbrales con F1 máximo, el punto medio del intervalo más ancho (máximo margen).
- Checkpoint en cada época: si un proceso se corta, continúa donde quedó.

In [ ]:
%%writefile train_job.py
import argparse, json, os, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
from timm.data import resolve_data_config
import torchvision.transforms as T
from sklearn.metrics import roc_auc_score, precision_recall_curve

CFG = json.load(open(os.environ['CACAO_CFG']))
OUT = CFG['paths']['out']
DTYPE = torch.bfloat16

# Estado de avance que lee la celda de monitoreo del notebook (logs/progress_w<proceso>.json)
PROG = {}
PROG_FILE = None


def prog(**kw):
    PROG.update(kw)
    PROG['t'] = time.time()
    if PROG_FILE:
        tmp = PROG_FILE + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(PROG, f)
        os.replace(tmp, PROG_FILE)


def ahora():
    return time.strftime('%H:%M:%S')


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def worker_init(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


# Imágenes en memoria (256 px, uint8), compartidas con los workers del DataLoader
_RAM = {}


def cargar(paths):
    for p in paths:
        if p not in _RAM:
            _RAM[p] = np.asarray(Image.open(p).convert('RGB'))
    return [_RAM[p] for p in paths]


class ImgDS(Dataset):
    def __init__(self, frame, tf):
        self.imgs = cargar(frame['cache_path'].tolist())
        self.y = frame['label'].astype('float32').tolist()
        self.tf = tf

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        return self.tf(Image.fromarray(self.imgs[i])), torch.tensor(self.y[i], dtype=torch.float32)


def build_transforms(model):
    dc = resolve_data_config({}, model=model)
    mean, std = dc['mean'], dc['std']
    s = CFG['img_size']
    a = CFG['aug']
    cj = a['color_jitter']
    train_tf = T.Compose([
        T.RandomResizedCrop(s, scale=(a['rrc_scale_min'], 1.0)),
        T.RandomHorizontalFlip(),
        T.RandomRotation(a['rotation_deg']),
        T.ColorJitter(cj, cj, cj),
        T.ToTensor(),
        T.Normalize(mean, std)])
    eval_tf = T.Compose([T.Resize((s, s)), T.ToTensor(), T.Normalize(mean, std)])
    return train_tf, eval_tf


def set_backbone_trainable(model, trainable):
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    for p in model.parameters():
        p.requires_grad = trainable or (id(p) in head_ids)


def bn_inferencia(model, phase):
    # BatchNorm en modo inferencia (estadísticas de ImageNet). Con bn_frozen también en la fase 2:
    # con lotes de 32 el recálculo desestabiliza la cabecera. gamma y beta se entrenan en la fase 2.
    if phase == 1 or CFG.get('bn_frozen', False):
        for m in model.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm):
                m.eval()


def run_epoch(model, loader, dev, opt=None, phase=1, etapa='train'):
    train = opt is not None
    model.train(train)
    if train:
        bn_inferencia(model, phase)
    crit = nn.BCEWithLogitsLoss()
    tot, n, ps, ys = 0.0, 0, [], []
    nb_total = len(loader)
    prog(etapa=etapa, lote=0, n_lotes=nb_total, loss_media=None)
    for b, (x, y) in enumerate(loader, start=1):
        x = x.to(dev, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(dev, non_blocking=True)
        with torch.set_grad_enabled(train), torch.autocast('cuda', dtype=DTYPE):
            logit = model(x).squeeze(1)
            loss = crit(logit.float(), y)
        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        tot += loss.item() * len(y)
        n += len(y)
        ps.append(torch.sigmoid(logit.float()).detach().cpu())
        ys.append(y.cpu())
        if b % 5 == 0 or b == nb_total:
            prog(lote=b, loss_media=tot / n)
    p = torch.cat(ps).numpy()
    yy = torch.cat(ys).numpy()
    acc = float(((p >= 0.5) == (yy == 1)).mean())
    auc = float(roc_auc_score(yy, p)) if len(np.unique(yy)) > 1 else float('nan')
    return tot / n, acc, auc


@torch.no_grad()
def predict(model, loader, dev):
    # Predicción final en float32, sin TF32
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    model.eval()
    ps = []
    for x, _ in loader:
        x = x.to(dev).to(memory_format=torch.channels_last)
        ps.append(torch.sigmoid(model(x).squeeze(1).float()).cpu())
    return torch.cat(ps).numpy()


def umbral_estable(y, p):
    # Umbral: punto medio del intervalo más ancho entre los que dan F1 máximo en validación
    # (el primer máximo variaba mucho entre pliegues con clases casi separadas)
    prec, rec, thr = precision_recall_curve(y, p)
    if not len(thr):
        return 0.5, 0.5, 0.5
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    optimos = thr[f1 >= f1.max() - 1e-9]
    ps = np.sort(np.unique(p))
    mejor = None
    for t in optimos:
        abajo = ps[ps < t]
        lo = float(abajo[-1]) if len(abajo) else 0.0
        if mejor is None or t - lo > mejor[1] - mejor[0]:
            mejor = (lo, float(t))
    return (mejor[0] + mejor[1]) / 2, mejor[0], mejor[1]


def atomic_save(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)


def train_one(job, dev):
    name, split, tag, seed = job['model'], job['split'], job['tag'], job['seed']
    arch = CFG['models'][name]
    done_flag = os.path.join(OUT, 'preds', tag + '.done')
    if os.path.exists(done_flag):
        return
    seed_all(seed)

    folds = pd.read_csv(os.path.join(OUT, 'folds.csv'))
    col = 'split_' + split
    tr, va, te = [folds[folds[col] == s].reset_index(drop=True) for s in ('train', 'val', 'test')]

    model = timm.create_model(arch, pretrained=True, num_classes=1, drop_rate=CFG['drop_rate'])
    # Inicialización común de la cabecera (timm usa U(-1, 1) en EfficientNet con una sola salida)
    head = model.get_classifier()
    nn.init.trunc_normal_(head.weight, std=0.02)
    nn.init.zeros_(head.bias)
    model = model.to(dev).to(memory_format=torch.channels_last)
    train_tf, eval_tf = build_transforms(model)
    g = torch.Generator()
    g.manual_seed(seed)
    nw = CFG['num_workers']
    kw = dict(batch_size=CFG['batch_size'], num_workers=nw, pin_memory=True,
              persistent_workers=nw > 0, worker_init_fn=worker_init)
    if nw > 0:
        kw['prefetch_factor'] = 4
    dl_tr = DataLoader(ImgDS(tr, train_tf), shuffle=True, drop_last=True, generator=g, **kw)
    dl_va = DataLoader(ImgDS(va, eval_tf), shuffle=False, **kw)
    dl_te = DataLoader(ImgDS(te, eval_tf), shuffle=False, **kw)

    ck_last = os.path.join(OUT, 'ckpt', tag + '_last.pt')
    ck_best = os.path.join(OUT, 'ckpt', tag + '_best.pt')
    st = {'phase': 1, 'epoch': 0, 'best_val': float('inf'), 'bad': 0, 'history': [],
          'opt': None, 'sched': None, 'train_seconds': 0.0, 'epochs_phase': {1: 0, 2: 0}}
    if os.path.exists(ck_last):
        st = torch.load(ck_last, map_location='cpu', weights_only=False)
        model.load_state_dict(st.pop('model'))
        print(ahora(), '[reanudando]', tag, 'fase', st['phase'], 'época', st['epoch'], flush=True)

    while st['phase'] <= 2:
        ph = st['phase']
        P = CFG['phase' + str(ph)]
        set_backbone_trainable(model, trainable=(ph == 2))
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=P['lr'], weight_decay=CFG['weight_decay'])
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', factor=CFG['plateau']['factor'],
            patience=CFG['plateau']['patience'], min_lr=CFG['plateau']['min_lr'])
        if st['opt'] is not None:
            opt.load_state_dict(st['opt'])
            sch.load_state_dict(st['sched'])
        while st['epoch'] < P['epochs'] and st['bad'] < P['patience']:
            t0 = time.time()
            prog(fase=ph, epoca=st['epoch'] + 1, max_epocas=P['epochs'], sin_mejora=st['bad'],
                 paciencia=P['patience'], mejor_val=None if st['best_val'] == float('inf') else st['best_val'])
            trl, tra, tru = run_epoch(model, dl_tr, dev, opt, ph, 'train')
            vl, vacc, vauc = run_epoch(model, dl_va, dev, None, ph, 'val')
            sch.step(vl)
            st['epoch'] += 1
            st['epochs_phase'][ph] += 1
            dt = time.time() - t0
            st['train_seconds'] += dt
            st['history'].append({'phase': ph, 'epoch_global': len(st['history']) + 1,
                                  'train_loss': trl, 'train_acc': tra, 'train_auc': tru,
                                  'val_loss': vl, 'val_acc': vacc, 'val_auc': vauc,
                                  'lr': opt.param_groups[0]['lr'], 'seconds': dt})
            if vl < st['best_val'] - 1e-4:
                st['best_val'] = vl
                st['bad'] = 0
                atomic_save(model.state_dict(), ck_best)
            else:
                st['bad'] += 1
            prog(ultima={'fase': ph, 'epoca': st['epoch'], 'train_loss': trl, 'train_acc': tra,
                         'val_loss': vl, 'val_acc': vacc, 'val_auc': vauc, 'seg': dt},
                 sin_mejora=st['bad'], mejor_val=st['best_val'])
            print('%s %s | fase %d | época %d/%d | train loss %.4f acc %.3f | val loss %.4f acc %.3f AUC %.4f'
                  ' | lr %.1e | sin mejora %d/%d | %.0f s' % (
                      ahora(), tag, ph, st['epoch'], P['epochs'], trl, tra, vl, vacc, vauc,
                      opt.param_groups[0]['lr'], st['bad'], P['patience'], dt), flush=True)
            st['opt'] = opt.state_dict()
            st['sched'] = sch.state_dict()
            atomic_save(dict(st, model=model.state_dict()), ck_last)
        # Fin de la fase: se recuperan los mejores pesos y se pasa a la siguiente
        model.load_state_dict(torch.load(ck_best, map_location=dev))
        st.update({'phase': ph + 1, 'epoch': 0, 'bad': 0, 'opt': None, 'sched': None})
        atomic_save(dict(st, model=model.state_dict()), ck_last)

    prog(etapa='predicción', lote=0, n_lotes=0)
    p_va = predict(model, dl_va, dev)
    p_te = predict(model, dl_te, dev)
    thr_val, thr_lo, thr_hi = umbral_estable(va['label'].values, p_va)
    partes = []
    for sp, frame, p in (('val', va, p_va), ('test', te, p_te)):
        o = frame[['image_path', 'label', 'group', 'fuente']].copy()
        o['prob'] = p
        o['split'] = sp
        o['particion'] = split
        o['model'] = name
        partes.append(o)
    pd.concat(partes).to_csv(os.path.join(OUT, 'preds', tag + '.csv'), index=False)
    pd.DataFrame(st['history']).to_csv(os.path.join(OUT, 'hist', tag + '.csv'), index=False)
    meta = {'model': name, 'arch': arch, 'particion': split, 'seed': seed,
            'thr_val': thr_val, 'thr_val_intervalo': [thr_lo, thr_hi],
            'best_val_loss': st['best_val'], 'epochs_phase1': st['epochs_phase'][1],
            'epochs_phase2': st['epochs_phase'][2], 'train_seconds': st['train_seconds'],
            'n_train': len(tr), 'n_val': len(va), 'n_test': len(te),
            'test_auc': float(roc_auc_score(te['label'].values, p_te))}
    with open(os.path.join(OUT, 'preds', tag + '.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    with open(done_flag, 'w') as f:
        f.write('ok')
    if os.path.exists(ck_last):
        os.remove(ck_last)
    print('%s [terminado] %s | AUC prueba %.4f | épocas %d+%d | %.1f min' % (
        ahora(), tag, meta['test_auc'], meta['epochs_phase1'], meta['epochs_phase2'],
        meta['train_seconds'] / 60), flush=True)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--worker', type=int, default=0)
    a = ap.parse_args()
    dev = torch.device('cuda')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_num_threads(2)              # varios procesos comparten los hilos de CPU
    jobs = json.load(open(os.path.join(OUT, 'jobs.json')))
    PROG_FILE = os.path.join(OUT, 'logs', 'progress_w%d.json' % a.worker)
    gpu = os.environ.get('CUDA_VISIBLE_DEVICES', '?')
    prog(worker=a.worker, gpu=gpu, estado='corriendo', inicio=time.time(), trabajo=None, n_hechos=0)
    print(ahora(), 'Proceso', a.worker, 'en GPU', gpu, flush=True)
    # Cola de trabajos: cada proceso reserva el siguiente con preds/<trabajo>.claim (O_EXCL)
    k = 0
    for job in jobs:
        tag = job['tag']
        if os.path.exists(os.path.join(OUT, 'preds', tag + '.done')):
            continue
        try:
            fd = os.open(os.path.join(OUT, 'preds', tag + '.claim'), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, str(a.worker).encode())
            os.close(fd)
        except FileExistsError:
            continue
        k += 1
        prog(i_trabajo=k, trabajo=tag, inicio_trabajo=time.time(), fase=None, epoca=None,
             ultima=None, etapa='cargando modelo', lote=0, n_lotes=0)
        print(ahora(), 'Proceso', a.worker, 'toma', tag, flush=True)
        try:
            train_one(job, dev)
            prog(n_hechos=k)
        except Exception as e:
            import traceback
            traceback.print_exc()
            print('[ERROR]', tag, repr(e), flush=True)
        torch.cuda.empty_cache()
    prog(estado='finalizado', etapa='-', trabajo=None)
    print(ahora(), 'Proceso', a.worker, 'finalizó.', flush=True)

In [ ]:
# Descarga previa de los pesos preentrenados
for nombre, arch in CFG['models'].items():
    m = timm.create_model(arch, pretrained=True, num_classes=1)
    print(nombre, arch, 'OK -', round(sum(p.numel() for p in m.parameters()) / 1e6, 2), 'M parámetros')
    del m

## 5. Entrenamiento

Lanza `PROCS_POR_GPU` procesos por GPU que toman trabajos de una cola (los más largos primero) y muestra el avance cada 5 s. Interrumpir la celda solo detiene la vista: `monitorear()` la vuelve a mostrar y `detener()` termina los procesos. Al relanzar, cada trabajo sigue desde su último checkpoint. Registro de cada proceso en `logs/worker<N>.log`.


In [ ]:
import signal
from IPython.display import clear_output

pid_file = OUT_DIR / 'logs' / 'pids.txt'

def vivo(pid):
    # Los procesos terminados quedan como zombis (estado Z) hasta que se recogen
    try:
        with open('/proc/%d/stat' % pid) as fh:
            return fh.read().rsplit(')', 1)[1].split()[0] != 'Z'
    except (OSError, IndexError):
        return False

def pids():
    return [int(x) for x in pid_file.read_text().split()] if pid_file.exists() else []

def procesos_vivos():
    return any(vivo(x) for x in pids())

def detener(espera=60):
    # Detiene todos los procesos; el trabajo en curso se reanuda desde su último checkpoint al relanzar.
    for x in pids():
        try:
            os.killpg(x, signal.SIGTERM)
        except OSError:
            pass
    t0 = time.time()
    while procesos_vivos() and time.time() - t0 < espera:
        time.sleep(1)
    for x in pids():
        if vivo(x):
            try:
                os.killpg(x, signal.SIGKILL)
            except OSError:
                pass
    print('Procesos detenidos.' if not procesos_vivos() else 'Algunos procesos siguen vivos; revisa con nvidia-smi.')

def lanzar():
    if procesos_vivos():
        print('Ya hay procesos corriendo (quizá de un lanzamiento anterior). Para relanzar con la configuración '
              'actual ejecuta detener() y luego esta celda otra vez. Se muestra el avance de los actuales.')
        return
    for f in list((OUT_DIR / 'preds').glob('*.claim')) + list((OUT_DIR / 'logs').glob('progress_w*.json')):
        f.unlink()                        # reservas y avances de lanzamientos anteriores
    procs = []
    for w in range(N_PROCS):
        gpu = GPUS[w % len(GPUS)]
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
        env['CACAO_CFG'] = str(CFG_PATH)
        log = open(OUT_DIR / 'logs' / ('worker' + str(w) + '.log'), 'a')
        p = subprocess.Popen([sys.executable, 'train_job.py', '--worker', str(w)],
                             stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
        procs.append(p)
        time.sleep(0.5)
    pid_file.write_text(' '.join(str(p.pid) for p in procs))
    print('Lanzados', N_PROCS, 'procesos:', PROCS_POR_GPU, 'por GPU en las GPU', GPUS)

def barra(x, n, ancho=24):
    if not n:
        return '[' + ' ' * ancho + ']'
    llenos = int(round(ancho * min(max(x, 0), n) / n))
    return '[' + '█' * llenos + '·' * (ancho - llenos) + ']'

def hms(s):
    s = int(max(s, 0))
    return '%d:%02d:%02d' % (s // 3600, s % 3600 // 60, s % 60)

def estado_gpu():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=index,utilization.gpu,memory.used,memory.total,temperature.gpu',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=10).stdout
        filas = {}
        for l in out.strip().splitlines():
            i, u, mu, mt, t = [x.strip() for x in l.split(',')]
            filas[int(i)] = 'uso %s%% | memoria %.1f/%.0f GB | %s °C' % (u, float(mu) / 1024, float(mt) / 1024, t)
        return filas
    except Exception:
        return {}

def monitorear(cada=5, recientes=8):
    jobs = [j['tag'] for j in json.load(open(JOBS_JSON))]
    t_ini = None
    try:
        while True:
            vivos = procesos_vivos()
            hechos = [t for t in jobs if (OUT_DIR / 'preds' / (t + '.done')).exists()]
            gpus = estado_gpu()
            avances = []
            for pf in sorted((OUT_DIR / 'logs').glob('progress_w*.json'), key=lambda x: int(x.stem.split('_w')[1])):
                try:
                    avances.append(json.load(open(pf)))
                except Exception:
                    pass
            inicios = [a['inicio'] for a in avances if 'inicio' in a]
            t_ini = min(inicios) if inicios else t_ini
            clear_output(wait=True)
            print('%s | %s | trabajos terminados %d de %d %s | transcurrido %s' % (
                time.strftime('%H:%M:%S'), 'PRUEBA RÁPIDA' if SMOKE_TEST else 'CORRIDA COMPLETA',
                len(hechos), len(jobs), barra(len(hechos), len(jobs)), hms(time.time() - t_ini) if t_ini else '-'))
            for g in GPUS:
                print('GPU %d | %s' % (g, gpus.get(g, 'nvidia-smi no disponible')))
            for a in avances:
                print('-' * 110)
                cab = 'proceso %d (GPU %s)' % (a.get('worker', -1), a.get('gpu', '?'))
                if a.get('estado') == 'finalizado':
                    print('%s | finalizado: %d trabajos | %s' % (cab, a.get('n_hechos', 0), hms(a['t'] - a['inicio'])))
                    continue
                linea = '%s | %s | %s en este trabajo' % (cab, a.get('trabajo'), hms(time.time() - a.get('inicio_trabajo', time.time())))
                if a.get('fase'):
                    mv = a.get('mejor_val')
                    linea += ' | fase %d, época %s/%s | sin mejora %s/%s | mejor val %s' % (
                        a['fase'], a.get('epoca'), a.get('max_epocas'), a.get('sin_mejora'), a.get('paciencia'),
                        '%.4f' % mv if mv is not None else '-')
                print(linea)
                lm = a.get('loss_media')
                l2 = '   %-15s %s %s/%s lotes | loss %s' % (a.get('etapa', ''), barra(a.get('lote', 0), a.get('n_lotes', 0)),
                                                        a.get('lote', 0), a.get('n_lotes', 0), '%.4f' % lm if lm is not None else '-')
                u = a.get('ultima')
                if u:
                    l2 += ' || última época: train %.4f/%.3f | val %.4f/%.3f AUC %.4f | %.0f s' % (
                        u['train_loss'], u['train_acc'], u['val_loss'], u['val_acc'], u['val_auc'], u['seg'])
                print(l2)
                if time.time() - a.get('t', time.time()) > 300:
                    print('   AVISO: sin actualizaciones hace más de 5 min; revisa logs/worker%d.log' % a.get('worker', -1))
            metas = sorted((OUT_DIR / 'preds').glob('*.json'), key=lambda x: x.stat().st_mtime)[-recientes:]
            if metas:
                print('-' * 110 + '\nÚltimos terminados:')
                for mf in reversed(metas):
                    try:
                        mt = json.load(open(mf))
                        print('   %-24s AUC prueba %.4f | épocas %d+%d | %.1f min' % (
                            mf.stem, mt['test_auc'], mt['epochs_phase1'],
                            mt['epochs_phase2'], mt['train_seconds'] / 60))
                    except Exception:
                        pass
            errores = []
            for lf in (OUT_DIR / 'logs').glob('worker*.log'):
                errores += [lf.name + ': ' + l.strip() for l in open(lf, errors='replace') if l.startswith('[ERROR]')]
            if errores:
                print('ERRORES:\n   ' + '\n   '.join(errores[-5:]))
            if not vivos:
                print('\nNo hay procesos corriendo. Terminados %d de %d.' % (len(hechos), len(jobs)))
                if len(hechos) < len(jobs):
                    print('Faltan trabajos: revisa logs/worker*.log y vuelve a ejecutar la celda de lanzamiento para reanudar.')
                break
            time.sleep(cada)
    except KeyboardInterrupt:
        clear_output(wait=True)
        print('Vista detenida. El entrenamiento SIGUE corriendo en las GPU; ejecuta monitorear() para volver a verlo.')

lanzar()
time.sleep(3)
monitorear()

In [ ]:
# Volver a mostrar el avance
monitorear()

In [ ]:
# Detener todos los entrenamientos (se reanudan con la celda de lanzamiento):

## 6. Métricas por pliegue y calibración

Requiere la corrida completa (`SMOKE_TEST = False`). Métricas de la clase BPR con umbral 0.5 y con el umbral de validación; Brier y ECE para la calibración. El Max F1 calculado sobre la prueba se guarda solo como referencia (es optimista).


In [ ]:
from scipy import stats
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, brier_score_loss,
                             precision_recall_curve, confusion_matrix, roc_curve)

class Aviso(Exception):
    # Se muestra como mensaje corto, sin traceback
    def _render_traceback_(self):
        return ['AVISO: ' + str(self)]

MODELOS = list(CFG['models'])
JOBS = json.load(open(JOBS_JSON))
faltan = [j['tag'] for j in JOBS if not (OUT_DIR / 'preds' / (j['tag'] + '.done')).exists()]
if faltan:
    raise Aviso('faltan %d de %d trabajos (%s).' % (
        len(faltan), len(JOBS), ', '.join(faltan[:5]) + (' ...' if len(faltan) > 5 else '')))
if SMOKE_TEST:
    raise Aviso('la prueba rápida solo entrena un pliegue; las secciones 6 a 12 se ejecutan con SMOKE_TEST = False.')

preds = pd.concat([pd.read_csv(OUT_DIR / 'preds' / (j['tag'] + '.csv')) for j in JOBS], ignore_index=True)
meta = pd.DataFrame([dict(json.load(open(OUT_DIR / 'preds' / (j['tag'] + '.json'))), tag=j['tag']) for j in JOBS])
es_cv = preds['particion'].str.startswith('r')
test = preds[(preds['split'] == 'test') & es_cv].merge(meta[['model', 'particion', 'thr_val']], on=['model', 'particion'])
test['repeticion'] = test['particion'].str.extract(r'r(\d+)_f').astype(int)[0] + 1
test['pliegue'] = test['particion'].str.extract(r'_f(\d+)').astype(int)[0] + 1

def metricas(y, p, thr):
    yp = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    return {'Accuracy': (tp + tn) / len(y), 'Precision': prec, 'Recall': rec,
            'Especificidad': tn / (tn + fp) if tn + fp else 0.0,
            'F1': 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
            'F1_macro': f1_score(y, yp, average='macro'),
            'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}

def ece(y, p, bins=10):
    # Error de calibración esperado: diferencia media (ponderada) entre confianza y frecuencia observada
    b = np.minimum((p * bins).astype(int), bins - 1)
    return sum(abs(p[b == k].mean() - y[b == k].mean()) * (b == k).mean() for k in range(bins) if (b == k).any())

filas = []
for (m, r, k), d in test.groupby(['model', 'repeticion', 'pliegue']):
    y, p = d['label'].values, d['prob'].values
    fila = {'Modelo': m, 'Repetición': r, 'Pliegue': k, 'AUC': roc_auc_score(y, p), 'AP': average_precision_score(y, p),
            'Brier': brier_score_loss(y, p), 'ECE': ece(y, p)}
    fila.update({key + '@0.5': v for key, v in metricas(y, p, 0.5).items()})
    fila.update({key + '@umbral_val': v for key, v in metricas(y, p, d['thr_val'].iloc[0]).items()})
    pr, rc, _ = precision_recall_curve(y, p)
    fila['MaxF1_prueba(optimista)'] = float(np.max(2 * pr * rc / np.clip(pr + rc, 1e-12, None)))
    filas.append(fila)
por_pliegue = pd.DataFrame(filas)
por_pliegue.to_csv(OUT_DIR / 'tablas' / 'metricas_por_pliegue.csv', index=False)

cols = ['AUC', 'AP', 'Accuracy@0.5', 'Precision@0.5', 'Recall@0.5', 'Especificidad@0.5', 'F1@0.5', 'F1_macro@0.5',
        'F1@umbral_val', 'MaxF1_prueba(optimista)', 'Brier', 'ECE']

def media_de(s):
    return '%.3f ± %.3f' % (s.mean(), s.std(ddof=1))

n_ej = por_pliegue.groupby('Modelo').size()
tabla_msd = por_pliegue.groupby('Modelo')[cols].agg(media_de).loc[MODELOS]
tabla_msd.insert(0, 'Ejecuciones', n_ej.loc[MODELOS].values)
# DE entre las medias de las tres repeticiones
med_rep = por_pliegue.groupby(['Modelo', 'Repetición'])['AUC'].mean().unstack()
tabla_msd['DE de la AUC entre repeticiones'] = med_rep.std(axis=1, ddof=1).loc[MODELOS].round(4) if med_rep.shape[1] > 1 else np.nan
tabla_msd.to_csv(OUT_DIR / 'tablas' / 'tabla_metricas_media_de.csv')
print('Media ± DE sobre %d ejecuciones por modelo (%d repeticiones × %d pliegues)' % (
    n_ej.iloc[0], CFG['n_repeats'], len(CFG['folds_to_run'])))
display(tabla_msd)
print('Umbral elegido en validación (media ± DE):')
display(meta[meta['particion'].str.startswith('r')].groupby('model')['thr_val'].agg(['mean', 'std', 'min', 'max']).loc[MODELOS].round(3))

## 7. Pruebas estadísticas sobre las predicciones fuera de pliegue

En cada repetición, cada imagen recibe exactamente una predicción de cada modelo, por lo que las comparaciones son pareadas sobre todas las imágenes. Se repiten en las 3 repeticiones para ver si las conclusiones se mantienen.

- AUC con IC95% y comparación de pares: DeLong et al. (1988), implementación rápida de Sun & Xu (2014).
- Exactitud (umbral 0.5): McNemar (exacta si hay menos de 25 discordancias).
- Corrección por comparaciones múltiples: Holm (1979), dentro de cada repetición.
- IC95% por bootstrap de grupos (2000 remuestreos) para exactitud y F1: las fotos de una misma secuencia se remuestrean juntas.
- Los contrastes son aproximados: las predicciones fuera de pliegue provienen de modelos entrenados con conjuntos solapados.

In [ ]:
from scipy.stats import norm
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from itertools import combinations

def _midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    Tm = np.zeros(N)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        Tm[i:j] = 0.5 * (i + j - 1)
        i = j
    out = np.empty(N)
    out[J] = Tm + 1
    return out

def fast_delong(preds_sorted, m):
    # preds_sorted: matriz (k modelos, n imágenes) con los m positivos en las primeras columnas
    n = preds_sorted.shape[1] - m
    k = preds_sorted.shape[0]
    tx = np.array([_midrank(preds_sorted[r, :m]) for r in range(k)])
    ty = np.array([_midrank(preds_sorted[r, m:]) for r in range(k)])
    tz = np.array([_midrank(preds_sorted[r, :]) for r in range(k)])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.atleast_2d(np.cov(v01)) / m + np.atleast_2d(np.cov(v10)) / n
    return aucs, cov

def delong(y, *ps):
    order = np.argsort(-y, kind='mergesort')
    return fast_delong(np.vstack(ps)[:, order], int(y.sum()))

# Predicciones fuera de pliegue: una por imagen y modelo en cada repetición
OOF = {}
for r in sorted(test['repeticion'].unique()):
    o = test[test['repeticion'] == r].pivot_table(index=['image_path', 'label'], columns='model', values='prob').reset_index()
    assert len(o) == len(df), 'Faltan predicciones fuera de pliegue en la repetición %d' % r
    OOF[r] = o

rng = np.random.default_rng(CFG['seed'])
B = 2000
filas_auc, filas_d, filas_m = [], [], []
for r, oof in OOF.items():
    y = oof['label'].values.astype(int)
    # Bootstrap por grupos: se remuestrean grupos completos (secuencias de captura)
    grp = oof['image_path'].map(df.set_index('image_path')['group']).values
    ug, gi = np.unique(grp, return_inverse=True)
    n_g = np.bincount(gi)
    gb = rng.integers(0, len(ug), size=(B, len(ug)))
    for mname in MODELOS:
        a, c = delong(y, oof[mname].values)
        se = np.sqrt(c[0, 0])
        yp = (oof[mname].values >= 0.5).astype(int)
        ok_g = np.bincount(gi, weights=((y == yp)).astype(float))
        tp_g = np.bincount(gi, weights=((y == 1) & (yp == 1)).astype(float))
        fp_g = np.bincount(gi, weights=((y == 0) & (yp == 1)).astype(float))
        fn_g = np.bincount(gi, weights=((y == 1) & (yp == 0)).astype(float))
        acc_b = ok_g[gb].sum(1) / n_g[gb].sum(1)
        tp_b, fp_b, fn_b = tp_g[gb].sum(1), fp_g[gb].sum(1), fn_g[gb].sum(1)
        f1_b = 2 * tp_b / np.clip(2 * tp_b + fp_b + fn_b, 1, None)
        filas_auc.append({'Repetición': r, 'Modelo': mname, 'AUC_OOF': a[0],
                          'IC95_inf': max(0.0, a[0] - 1.96 * se), 'IC95_sup': min(1.0, a[0] + 1.96 * se),
                          'Accuracy_OOF': (yp == y).mean(),
                          'Accuracy_IC95': '%.3f–%.3f' % tuple(np.percentile(acc_b, [2.5, 97.5])),
                          'F1_OOF': f1_score(y, yp),
                          'F1_IC95': '%.3f–%.3f' % tuple(np.percentile(f1_b, [2.5, 97.5])),
                          'Brier_OOF': brier_score_loss(y, oof[mname].values)})
    pd_r, pm_r = [], []
    for a_, b_ in combinations(MODELOS, 2):
        aucs, c = delong(y, oof[a_].values, oof[b_].values)
        var = c[0, 0] + c[1, 1] - 2 * c[0, 1]
        z = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else 0.0
        pd_r.append({'Repetición': r, 'Modelo A': a_, 'Modelo B': b_, 'AUC A': aucs[0], 'AUC B': aucs[1],
                     'Diferencia': aucs[0] - aucs[1], 'z': z, 'p': 2 * norm.sf(abs(z))})
        ca = (oof[a_].values >= 0.5).astype(int) == y
        cb = (oof[b_].values >= 0.5).astype(int) == y
        tab = [[int((ca & cb).sum()), int((ca & ~cb).sum())], [int((~ca & cb).sum()), int((~ca & ~cb).sum())]]
        disc = tab[0][1] + tab[1][0]
        res = mcnemar(tab, exact=disc < 25, correction=True)
        pm_r.append({'Repetición': r, 'Modelo A': a_, 'Modelo B': b_, 'Accuracy A': ca.mean(), 'Accuracy B': cb.mean(),
                     'Solo A acierta': tab[0][1], 'Solo B acierta': tab[1][0], 'Estadístico': res.statistic, 'p': res.pvalue})
    pd_r, pm_r = pd.DataFrame(pd_r), pd.DataFrame(pm_r)
    pd_r['p_Holm'] = multipletests(pd_r['p'], method='holm')[1]
    pm_r['p_Holm'] = multipletests(pm_r['p'], method='holm')[1]
    filas_d.append(pd_r)
    filas_m.append(pm_r)

tabla_auc = pd.DataFrame(filas_auc)
tabla_delong = pd.concat(filas_d, ignore_index=True)
tabla_mcnemar = pd.concat(filas_m, ignore_index=True)

# Resumen por par en las tres repeticiones
def resumen_pares(t, col_dif):
    g = t.groupby(['Modelo A', 'Modelo B'])
    return pd.DataFrame({'Diferencia media': g[col_dif].mean(), 'p_Holm mínimo': g['p_Holm'].min(),
                         'p_Holm máximo': g['p_Holm'].max(),
                         'Repeticiones con p_Holm < 0.05': g['p_Holm'].apply(lambda s: '%d de %d' % ((s < 0.05).sum(), len(s)))}).reset_index()

tabla_mcnemar['Diferencia de accuracy'] = tabla_mcnemar['Accuracy A'] - tabla_mcnemar['Accuracy B']
resumen_delong = resumen_pares(tabla_delong, 'Diferencia')
resumen_mcnemar = resumen_pares(tabla_mcnemar, 'Diferencia de accuracy')


tabla_auc.to_csv(OUT_DIR / 'tablas' / 'tabla_auc_oof_ic95.csv', index=False)
tabla_delong.to_csv(OUT_DIR / 'tablas' / 'tabla_delong_pares.csv', index=False)
tabla_mcnemar.to_csv(OUT_DIR / 'tablas' / 'tabla_mcnemar_pares.csv', index=False)
resumen_delong.to_csv(OUT_DIR / 'tablas' / 'resumen_delong_repeticiones.csv', index=False)
resumen_mcnemar.to_csv(OUT_DIR / 'tablas' / 'resumen_mcnemar_repeticiones.csv', index=False)
print('AUC fuera de pliegue con IC95% (DeLong), por repetición:')
display(tabla_auc.round(4))
print('DeLong por pares, resumen de las repeticiones (corrección de Holm dentro de cada repetición):')
display(resumen_delong.round(4))
print('McNemar por pares, resumen de las repeticiones:')
display(resumen_mcnemar.round(4))

## 8. Figuras

Repetición 1. Las matrices de confusión usan las predicciones fuera de pliegue y suman las 2211 imágenes.


In [ ]:
import seaborn as sns
from sklearn.calibration import calibration_curve
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'], 'font.size': 10})
R1 = 1                                     # repetición usada en las figuras
oof = OOF[R1]
y = oof['label'].values.astype(int)
ta = tabla_auc[tabla_auc['Repetición'] == R1].set_index('Modelo')

# Curvas ROC fuera de pliegue
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
for mname in MODELOS:
    fpr, tpr, _ = roc_curve(y, oof[mname].values)
    ax.plot(fpr, tpr, lw=1.6, label='%s: AUC = %.3f (IC95%% %.3f–%.3f)' % (
        mname, ta.loc[mname, 'AUC_OOF'], ta.loc[mname, 'IC95_inf'], ta.loc[mname, 'IC95_sup']))
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_xlabel('Tasa de falsos positivos (1 - especificidad)')
ax.set_ylabel('Tasa de verdaderos positivos (sensibilidad)')
ax.legend(loc='lower right', fontsize=7)
ax.grid(alpha=0.3)
fig.savefig(OUT_DIR / 'figs' / 'fig_roc_oof.png', bbox_inches='tight')
plt.show()

# Matrices de confusión fuera de pliegue (umbral 0.5); todas suman el total de imágenes
fig, axes = plt.subplots(1, len(MODELOS), figsize=(3.3 * len(MODELOS), 3.2), dpi=300)
for ax, mname in zip(axes, MODELOS):
    cm = confusion_matrix(y, (oof[mname].values >= 0.5).astype(int), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greys', cbar=False, ax=ax,
                xticklabels=CFG['class_names'], yticklabels=CFG['class_names'])
    ax.set_title(mname + ' (n = ' + str(cm.sum()) + ')', fontsize=9)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
plt.tight_layout()
fig.savefig(OUT_DIR / 'figs' / 'fig_matrices_confusion_oof.png', bbox_inches='tight')
plt.show()

# Curvas de entrenamiento y validación de los 5 pliegues de la repetición 1 (línea punteada: inicio de la fase 2)
fig, axes = plt.subplots(2, len(MODELOS), figsize=(3.3 * len(MODELOS), 5.5), dpi=300, sharey='row')
for j, mname in enumerate(MODELOS):
    for k in CFG['folds_to_run']:
        h = pd.read_csv(OUT_DIR / 'hist' / ('%s_r%d_f%d.csv' % (mname, R1 - 1, k)))
        l, = axes[0, j].plot(h['epoch_global'], h['val_loss'], lw=0.9, label='Pliegue %d (val)' % (k + 1))
        axes[0, j].plot(h['epoch_global'], h['train_loss'], lw=0.6, ls='--', color=l.get_color(), alpha=0.6)
        axes[1, j].plot(h['epoch_global'], h['val_auc'], lw=0.9)
        corte = int((h['phase'] == 1).sum()) + 0.5
        axes[0, j].axvline(corte, color='grey', ls=':', lw=0.6)
        axes[1, j].axvline(corte, color='grey', ls=':', lw=0.6)
    axes[0, j].set_title(mname, fontsize=9)
    axes[1, j].set_xlabel('Época')
axes[0, 0].set_ylabel('Pérdida BCE (— val, -- entren.)')
axes[1, 0].set_ylabel('AUC de validación')
axes[0, 0].legend(fontsize=6)
plt.tight_layout()
fig.savefig(OUT_DIR / 'figs' / 'fig_curvas_entrenamiento.png', bbox_inches='tight')
plt.show()

# Diagrama de calibración (fiabilidad) fuera de pliegue
fig, ax = plt.subplots(figsize=(5.5, 5.5), dpi=300)
for mname in MODELOS:
    fo, mp = calibration_curve(y, oof[mname].values, n_bins=10, strategy='quantile')
    ax.plot(mp, fo, marker='o', ms=3, lw=1.2, label='%s (Brier %.3f)' % (mname, ta.loc[mname, 'Brier_OOF']))
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_xlabel('Probabilidad predicha de BPR')
ax.set_ylabel('Frecuencia observada de BPR')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
fig.savefig(OUT_DIR / 'figs' / 'fig_calibracion_oof.png', bbox_inches='tight')
plt.show()

## 9. Cambio de estrato de resolución (se omite si una clase tiene menos de 20 imágenes)

In [ ]:
# Cambio de estrato: entrenar con un tamaño de imagen y evaluar con el otro
src = preds[(preds['split'] == 'test') & preds['particion'].str.startswith('src_')]
if not len(src):
    print('No hay trabajos de cambio de fuente (desactivados o sin imágenes suficientes por clase).')
else:
    auc_cv = por_pliegue.groupby('Modelo')['AUC'].mean()
    filas_s = []
    for (m, s), d in src.groupby(['model', 'particion']):
        yv, pv = d['label'].values, d['prob'].values
        mt = meta[(meta['model'] == m) & (meta['particion'] == s)].iloc[0]
        ent = s[-1]
        pru = 'B' if ent == 'A' else 'A'
        fila = {'Modelo': m, 'Entrena': 'fuente %s (%s)' % (ent, fuentes[0 if ent == 'A' else 1]),
                'Prueba': 'fuente %s (%s)' % (pru, fuentes[0 if pru == 'A' else 1]),
                'n entren.+val': int(mt['n_train'] + mt['n_val']), 'n prueba': len(d),
                'AUC': roc_auc_score(yv, pv), 'AUC en validación cruzada': auc_cv[m]}
        fila['Caída de AUC'] = fila['AUC en validación cruzada'] - fila['AUC']
        fila.update({k + '@0.5': v for k, v in metricas(yv, pv, 0.5).items() if k in ('Accuracy', 'Recall', 'Especificidad', 'F1_macro')})
        filas_s.append(fila)
    tabla_fuente = pd.DataFrame(filas_s).sort_values(['Entrena', 'Modelo'])
    tabla_fuente.to_csv(OUT_DIR / 'tablas' / 'tabla_cambio_fuente.csv', index=False)
    display(tabla_fuente.round(4))

## 10. Complejidad computacional

Con las GPU libres. Cada modelo se mide en un proceso nuevo, tres veces (media ± DE); la memoria es el pico asignado por PyTorch con lote 1. FLOPs con `FlopCounterMode` (una multiplicación-suma cuenta como 2 FLOPs).


In [ ]:
%%writefile bench_job.py
import json, sys, time
import torch, timm
from torch.utils.flop_counter import FlopCounterMode
arch, S = sys.argv[1], int(sys.argv[2])
dev = torch.device('cuda:0')

def latencia_gpu(model, x, n=200, warm=30, amp=False):
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16, enabled=amp):
        for _ in range(warm):
            model(x)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n):
            model(x)
        torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
m = timm.create_model(arch, pretrained=False, num_classes=1).eval().to(dev)
x1 = torch.randn(1, 3, S, S, device=dev)
with torch.no_grad(), FlopCounterMode(display=False) as fc:
    m(x1)
gflops = fc.get_total_flops() / 1e9
torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(dev)
ms_fp32 = latencia_gpu(m, x1)
mem_mb = torch.cuda.max_memory_allocated(dev) / 1024**2
ms_bf16 = latencia_gpu(m, x1, amp=True)
xb = torch.randn(64, 3, S, S, device=dev)
img_s = 64 / (latencia_gpu(m, xb, n=30, warm=5, amp=True) / 1000)
del xb
m = m.cpu(); torch.set_num_threads(4)
xc = torch.randn(1, 3, S, S)
with torch.no_grad():
    for _ in range(5):
        m(xc)
    t0 = time.time()
    for _ in range(30):
        m(xc)
ms_cpu = (time.time() - t0) / 30 * 1000
npar = sum(p.numel() for p in m.parameters())
print(json.dumps({'npar': npar, 'gflops': gflops, 'ms_fp32': ms_fp32, 'ms_bf16': ms_bf16, 'img_s': img_s,
                  'mem_mb': mem_mb, 'ms_cpu': ms_cpu}))


In [ ]:
# Cada modelo en un proceso limpio, 3 repeticiones (GPU libres)
REP = 3
filas_c = []
for mname, arch in CFG['models'].items():
    rs = []
    for k in range(REP):
        out = subprocess.run([sys.executable, 'bench_job.py', arch, str(CFG['img_size'])], capture_output=True, text=True,
                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(GPUS[0])))
        assert out.returncode == 0, out.stderr[-2000:]
        rs.append(json.loads(out.stdout.strip().splitlines()[-1]))
        print('%s repetición %d: GPU FP32 %.2f ms | CPU %.1f ms' % (mname, k + 1, rs[-1]['ms_fp32'], rs[-1]['ms_cpu']), flush=True)
    r = pd.DataFrame(rs)
    ms = lambda c, d=2: '%.*f ± %.*f' % (d, r[c].mean(), d, r[c].std())
    filas_c.append({'Modelo': mname, 'Arquitectura (timm)': arch,
                    'Parámetros (M)': round(r['npar'].iloc[0] / 1e6, 2),
                    'Tamaño FP32 (MB)': round(r['npar'].iloc[0] * 4 / 1024**2, 1),
                    'GFLOPs (224×224)': round(r['gflops'].iloc[0], 2),
                    'Latencia GPU FP32 (ms/img)': ms('ms_fp32'),
                    'Latencia GPU BF16 (ms/img)': ms('ms_bf16'),
                    'Rendimiento GPU BF16 (img/s, lote 64)': '%.0f ± %.0f' % (r['img_s'].mean(), r['img_s'].std()),
                    'Memoria GPU pico (MB, lote 1)': round(r['mem_mb'].mean()),
                    'Latencia CPU 4 hilos (ms/img)': ms('ms_cpu', 1)})
tabla_complejidad = pd.DataFrame(filas_c)
tabla_complejidad.to_csv(OUT_DIR / 'tablas' / 'tabla_complejidad.csv', index=False)
display(tabla_complejidad)


## 11. Grad-CAM

Se usan los modelos de la repetición 1, pliegue 1, sobre imágenes de su conjunto de prueba. El mapa explica la clase predicha (logit para BPR, logit negativo para Sana). Capas objetivo: ResNet50 `layer4`, ConvNeXt última etapa, EfficientNetV2 `conv_head`, Swin norma del último bloque (tokens 7×7), DeiT norma del último bloque (parches 14×14, sin el token de clase).

In [ ]:
import torchvision.transforms as T
from timm.data import resolve_data_config
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

class ObjetivoBinario:
    # clase 1 = BPR (logit); clase 0 = Sana (-logit)
    def __init__(self, clase):
        self.signo = 1.0 if clase == 1 else -1.0
    def __call__(self, salida):
        return self.signo * salida.sum()

def reshape_swin(t):
    if t.ndim == 4:
        return t.permute(0, 3, 1, 2)
    b, l, c = t.shape
    s = int(round(l ** 0.5))
    return t.reshape(b, s, s, c).permute(0, 3, 1, 2)

def capas_objetivo(nombre, m):
    arch = CFG['models'][nombre]
    if arch.startswith('resnet'):
        return [m.layer4[-1]], None
    if arch.startswith('convnext'):
        return [m.stages[-1]], None
    if 'efficientnet' in arch:
        return [m.conv_head], None
    if arch.startswith('swin'):
        return [m.layers[-1].blocks[-1].norm2], reshape_swin
    if arch.startswith('deit') or arch.startswith('vit'):
        npt = m.num_prefix_tokens
        def reshape_vit(t):
            t = t[:, npt:, :]
            b, l, c = t.shape
            s = int(round(l ** 0.5))
            return t.reshape(b, s, s, c).permute(0, 3, 1, 2)
        return [m.blocks[-1].norm1], reshape_vit
    raise ValueError(arch)

def cargar_modelo(nombre, particion, dev):
    m = timm.create_model(CFG['models'][nombre], pretrained=False, num_classes=1, drop_rate=CFG['drop_rate'])
    sd = torch.load(OUT_DIR / 'ckpt' / (nombre + '_' + particion + '_best.pt'), map_location='cpu')
    m.load_state_dict(sd)
    m.eval().to(dev)
    for p in m.parameters():
        p.requires_grad_(True)
    return m

def tf_eval(m):
    dc = resolve_data_config({}, model=m)
    return T.Compose([T.Resize((CFG['img_size'], CFG['img_size'])), T.ToTensor(), T.Normalize(dc['mean'], dc['std'])])

PART_CAM = 'r0_f0'                         # repetición 1, pliegue 1
te0 = df[df['split_' + PART_CAM] == 'test']
rng_cam = np.random.default_rng(CFG['seed'])
sel = (list(rng_cam.choice(te0[te0['label'] == 1].index, 3, replace=False)) +
       list(rng_cam.choice(te0[te0['label'] == 0].index, 3, replace=False)))
mejor = tabla_auc[tabla_auc['Repetición'] == 1].sort_values('AUC_OOF', ascending=False)['Modelo'].iloc[0]
p_mejor = test[(test['model'] == mejor) & (test['particion'] == PART_CAM)].set_index('image_path')['prob']
errores = [i for i in te0.index
           if (p_mejor.loc[df.loc[i, 'image_path']] >= 0.5) != (df.loc[i, 'label'] == 1)][:4]

def figura_cam(indices, archivo, titulo):
    dev = torch.device('cuda:0')
    filas, cols = len(indices), len(MODELOS) + 1
    fig, axes = plt.subplots(filas, cols, figsize=(2.1 * cols, 2.3 * filas), dpi=300)
    axes = np.atleast_2d(axes)
    base = [np.asarray(Image.open(df.loc[i, 'cache_path']).convert('RGB')
                       .resize((CFG['img_size'], CFG['img_size']))).astype(np.float32) / 255.0 for i in indices]
    for r, i in enumerate(indices):
        axes[r, 0].imshow(base[r])
        axes[r, 0].set_title('Real: ' + CFG['class_names'][int(df.loc[i, 'label'])], fontsize=7)
        axes[r, 0].axis('off')
    for c, nombre in enumerate(MODELOS, start=1):
        m = cargar_modelo(nombre, PART_CAM, dev)
        tf = tf_eval(m)
        capas, rt = capas_objetivo(nombre, m)
        with GradCAM(model=m, target_layers=capas, reshape_transform=rt) as cam:
            for r, i in enumerate(indices):
                x = tf(Image.open(df.loc[i, 'cache_path']).convert('RGB')).unsqueeze(0).to(dev)
                with torch.no_grad():
                    prob = torch.sigmoid(m(x)).item()
                pred = int(prob >= 0.5)
                mapa = cam(input_tensor=x, targets=[ObjetivoBinario(pred)])[0]
                axes[r, c].imshow(show_cam_on_image(base[r], mapa, use_rgb=True))
                axes[r, c].set_title(nombre + ': ' + CFG['class_names'][pred] + ' (p=%.2f)' % prob, fontsize=6)
                axes[r, c].axis('off')
        del m
        torch.cuda.empty_cache()
    fig.suptitle(titulo, fontsize=8)
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'figs' / archivo, bbox_inches='tight')
    plt.show()

figura_cam(sel, 'fig_gradcam_ejemplos.png', 'Grad-CAM (repetición 1, pliegue 1): 3 mazorcas con BPR y 3 sanas')
if errores:
    figura_cam(errores, 'fig_gradcam_errores.png', 'Grad-CAM: casos mal clasificados por ' + mejor + ' (repetición 1, pliegue 1)')

## 12. Tabla de hiperparámetros, exportación de tablas y requirements

In [ ]:
filas_h = []
for mname, arch in CFG['models'].items():
    mm = meta[(meta['model'] == mname) & meta['particion'].str.startswith('r')]
    filas_h.append({
        'Modelo': mname, 'Arquitectura (timm)': arch, 'Pesos iniciales': 'ImageNet-1k',
        'Entrada': '%d×%d px' % (CFG['img_size'], CFG['img_size']),
        'Cabecera': 'Pooling global + Dropout(%.1f) + Lineal(1 salida) + sigmoide' % CFG['drop_rate'],
        'BatchNorm': 'modo inferencia en ambas fases' if CFG['bn_frozen'] else 'modo inferencia solo en fase 1',
        'Inicialización de la capa de salida': 'normal truncada (DE 0.02), sesgo 0',
        'Fase 1': 'backbone congelado; AdamW lr=%g; máx. %d épocas; paciencia %d' % (
            CFG['phase1']['lr'], CFG['phase1']['epochs'], CFG['phase1']['patience']),
        'Fase 2': 'modelo completo; AdamW lr=%g; máx. %d épocas; paciencia %d' % (
            CFG['phase2']['lr'], CFG['phase2']['epochs'], CFG['phase2']['patience']),
        'Weight decay': CFG['weight_decay'], 'Lote (ambas fases)': CFG['batch_size'],
        'Épocas ejecutadas fase 1 (media)': round(mm['epochs_phase1'].mean(), 1),
        'Épocas ejecutadas fase 2 (media)': round(mm['epochs_phase2'].mean(), 1),
        'Umbral en validación': 'punto medio del intervalo más ancho con F1 máximo',
        'Tiempo por pliegue (min, media ± DE)': '%.1f ± %.1f' % (
            mm['train_seconds'].mean() / 60, mm['train_seconds'].std() / 60)})
tabla_hiper = pd.DataFrame(filas_h)
tabla_hiper.to_csv(OUT_DIR / 'tablas' / 'tabla_hiperparametros.csv', index=False)
display(tabla_hiper)

hojas = {'Conteos': tabla_conteos, 'Metricas_media_DE': tabla_msd.reset_index(),
         'Metricas_por_pliegue': por_pliegue, 'AUC_OOF_IC95': tabla_auc, 'DeLong': tabla_delong,
         'McNemar': tabla_mcnemar, 'Resumen_DeLong': resumen_delong, 'Resumen_McNemar': resumen_mcnemar,
         'Hiperparametros': tabla_hiper}
for nombre_t, var in (('Cambio_fuente', 'tabla_fuente'), ('Control_atajos', 'tabla_atajos')):
    if var in globals():
        hojas[nombre_t] = globals()[var]
if 'tabla_complejidad' in globals():
    hojas['Complejidad'] = tabla_complejidad
with pd.ExcelWriter(OUT_DIR / 'tablas' / 'tablas_articulo.xlsx') as xw:
    for nombre, t in hojas.items():
        t.to_excel(xw, sheet_name=nombre, index=False)

pkgs = ['torch', 'torchvision', 'timm', 'grad-cam', 'scikit-learn', 'scipy', 'statsmodels', 'numpy',
        'pandas', 'Pillow', 'ImageHash', 'matplotlib', 'seaborn', 'openpyxl']
with open(OUT_DIR / 'requirements.txt', 'w') as f:
    for p in pkgs:
        try:
            f.write(p + '==' + md.version(p) + chr(10))
        except Exception:
            pass
print('Tablas en', OUT_DIR / 'tablas', '| figuras en', OUT_DIR / 'figs')